In [ ]:
# Mount Google Drive (if in Colab environment)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Necessary Libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error
import joblib

# Load the dataset
file_path = '/content/drive/MyDrive/windforecast/df_final_features.feather'
df = pd.read_feather(file_path)

# Initial Data Checks
print("Shape:", df.shape)
df.set_index('date_time', inplace=True)
print("Index:", df.index)  # date_time, daily frequency
print("Number of NaN values:", df.isna().sum().sum())  # Should be 0
print("Is Target Column Present?:", 'target' in df.columns)  # Should be True
print("Target Preview:", df['target'].head())  # Should show mean_wind_speed.shift(-1)
print("Index Frequency:", pd.infer_freq(df.index))  # Should be 'D'

# Ensure the index is datetime and has daily frequency
df.index = pd.to_datetime(df.index)
assert df.index.freq == 'D' or pd.infer_freq(df.index) == 'D', "Index is not daily frequency!"

In [ ]:
del df['mean_wind_speed_dev_from_ema_3']

In [ ]:
df.max().max(), df.min().min()

In [ ]:
df.max().sort_values()

In [ ]:
df['sarkoy_temp_max_to_mean_ratio'].sort_values()

In [ ]:
df.info()

In [ ]:
from sklearn.preprocessing import RobustScaler, MinMaxScaler, StandardScaler, MaxAbsScaler

# Cast df int types to float64
df = df.astype('float64')

# Train-Val-Test Split (2023 Val, 2024 Test)
train_df = df[df.index.year < 2023]
val_df = df[df.index.year == 2023]
test_df = df[df.index.year == 2024]
print("Train seti:", train_df.index.min(), "to", train_df.index.max(), f"({len(train_df)} rows)")
print("Validation seti:", val_df.index.min(), "to", val_df.index.max(), f"({len(val_df)} rows)")
print("Test seti:", test_df.index.min(), "to", test_df.index.max(), f"({len(test_df)} rows)")

# Define columns to drop
cols_to_drop = ['target', 'persistence_pred', 'seasonal_naive_pred', 'year', 'day_of_year']
# Drop columns if they exist in the DataFrame
X_train = train_df.drop(columns=[col for col in cols_to_drop if col in train_df.columns])
y_train = train_df['target']
X_val = val_df.drop(columns=[col for col in cols_to_drop if col in val_df.columns])
y_val = val_df['target']
X_test = test_df.drop(columns=[col for col in cols_to_drop if col in test_df.columns])
y_test = test_df['target']

# Combine training and validation sets for cross-validation
X_train_cv = pd.concat([X_train, X_val])
y_train_cv = pd.concat([y_train, y_val])

# Monthly-based scalers
scalers = {}
months = range(1, 13)  # 1: January, 12: December
for month in months:
    scalers[f'scaler_{month}'] = StandardScaler()

# Fit scalers and transform the training set and train_cv set
X_train_scaled = X_train.copy()
X_train_cv_scaled = X_train_cv.copy()

for month in months:
    month_mask = X_train.index.month == month
    cv_month_mask = X_train_cv.index.month == month

    if month_mask.sum() > 0:  # If there's data for the month in the training set
        X_train_scaled.loc[month_mask, X_train.columns] = scalers[f'scaler_{month}'].fit_transform(X_train.loc[month_mask, X_train.columns]).astype('float64')

    if cv_month_mask.sum() > 0:  # Fit and transform the combined set
        X_train_cv_scaled.loc[cv_month_mask, X_train_cv.columns] = scalers[f'scaler_{month}'].fit_transform(X_train_cv.loc[cv_month_mask, X_train_cv.columns]).astype('float64')

# Apply scalers to validation and test sets
X_val_scaled = X_val.copy()
X_test_scaled = X_test.copy()
for month in months:
    val_month_mask = X_val.index.month == month
    test_month_mask = X_test.index.month == month
    if val_month_mask.sum() > 0:
        X_val_scaled.loc[val_month_mask, X_val.columns] = scalers[f'scaler_{month}'].transform(X_val.loc[val_month_mask, X_val.columns]).astype('float64')
    if test_month_mask.sum() > 0:
        X_test_scaled.loc[test_month_mask, X_test.columns] = scalers[f'scaler_{month}'].transform(X_test.loc[test_month_mask, X_test.columns]).astype('float64')

# Store scaled data in DataFrames
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_train_cv_scaled = pd.DataFrame(X_train_cv_scaled, columns=X_train_cv.columns, index=X_train_cv.index)
X_val_scaled = pd.DataFrame(X_val_scaled, columns=X_val.columns, index=X_val.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

# Verify outputs
print("Scaled X_train shape:", X_train_scaled.shape)
print("Scaled X_train_cv shape:", X_train_cv_scaled.shape)
print("Scaled X_val shape:", X_val_scaled.shape)
print("Scaled X_test shape:", X_test_scaled.shape)
print("X_train_scaled NaN count:", X_train_scaled.isna().sum().sum())
print("X_train_cv_scaled NaN count:", X_train_cv_scaled.isna().sum().sum())
print("X_val_scaled NaN count:", X_val_scaled.isna().sum().sum())
print("X_test_scaled NaN count:", X_test_scaled.isna().sum().sum())

# Save scalers
import joblib
import os

# Create directory if it does not exist
scaler_dir = '/content/drive/MyDrive/windforecast/scalers'
os.makedirs(scaler_dir, exist_ok=True)

for month in months:
    joblib.dump(scalers[f'scaler_{month}'], f'{scaler_dir}/scaler_{month}.pkl')

print("Scalers saved: /content/drive/MyDrive/windforecast/scalers/scaler_1.pkl, ..., scaler_12.pkl")

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.linear_model import LassoCV, RidgeCV, ElasticNetCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, max_error
from sklearn.model_selection import TimeSeriesSplit
import joblib

# Model evaluation function
def evaluate_model(y_true, y_pred, model_name, dataset="Validation"):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    max_ae = max_error(y_true, y_pred)
    print(f"\n{model_name} Performance ({dataset} Set):")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"R²: {r2:.4f}")
    print(f"MaxAE: {max_ae:.4f}")
    return {'Model': model_name, 'Dataset': dataset, 'RMSE': rmse, 'MAE': mae, 'R²': r2, 'MaxAE': max_ae}

In [ ]:
# Create directories
base_dir = '/content/drive/MyDrive/windforecast/'
predictions_dir = os.path.join(base_dir, 'predictions')
models_dir = os.path.join(base_dir, 'models')
os.makedirs(predictions_dir, exist_ok=True)
os.makedirs(models_dir, exist_ok=True)
print(f"Predictions directory: {predictions_dir}")
print(f"Models directory: {models_dir}")

# NaN check
print("X_train_scaled NaN count:", X_train_scaled.isna().sum().sum())
print("X_val_scaled NaN count:", X_val_scaled.isna().sum().sum())
print("X_test_scaled NaN count:", X_test_scaled.isna().sum().sum())

In [ ]:
for df_temp in [X_train_scaled, X_val_scaled, X_test_scaled]:
    assert df_temp.isna().sum().sum() == 0, "NaN values found in scaled data!"
    print("No NaN values found in scaled data!")
    print(df_temp.shape)
    print(df_temp.max().max(), df_temp.min().min())

In [ ]:
for df_temp in [X_train_scaled, X_val_scaled, X_test_scaled]:
    assert df_temp.isna().sum().sum() == 0, "NaN values found in scaled data!"
    print("No NaN values found in scaled data!")
    print(df_temp.shape)
    print(df_temp.max().max(), df_temp.min().min())

# End OF DATA LOAD and PREPROCESS

In [ ]:
# Optimization Study of Linear Models
alphas = [0.001, 0.003, 0.005, 0.007, 0.01, 0.03, 0.05, 0.07, 0.1, 0.3, 0.5, 0.7, 1, 3, 5, 7, 10, 15, 20, 25, 30, 50, 100]
l1_ratios = [0.1, 0.3, 0.5, 0.7, 0.9]

# Import and create time series split with cv = 6
tscv = TimeSeriesSplit(n_splits=6)

# Define models
models = {
    'LassoCV': LassoCV(cv=tscv, alphas=alphas, random_state=42, max_iter=200000),
    'RidgeCV': RidgeCV(cv=tscv, alphas=alphas),
    'ElasticNetCV': ElasticNetCV(cv=tscv, alphas=alphas, l1_ratio=l1_ratios, random_state=42, max_iter=200000)
}

results = []

for model_name, model in models.items():
    print(f"\nEvaluating {model_name}...")
    # Train model
    model.fit(X_train_scaled, y_train)

    # Training set prediction and evaluation
    y_train_pred = model.predict(X_train_scaled)
    train_results = evaluate_model(y_train, y_train_pred, model_name, "Train")
    results.append(train_results)
    # Save training predictions
    train_predictions_df = pd.DataFrame({
        'date_time': X_train.index,
        'actual': y_train,
        'predicted': y_train_pred
    })
    train_predictions_path = os.path.join(predictions_dir, f'{model_name.lower()}_train_predictions.csv')
    train_predictions_df.to_csv(train_predictions_path, index=False)
    print(f"{model_name} training predictions saved: {train_predictions_path}")

    # Validation set prediction and evaluation
    y_val_pred = model.predict(X_val_scaled)
    val_results = evaluate_model(y_val, y_val_pred, model_name, "Validation")
    results.append(val_results)
    # Save validation predictions
    val_predictions_df = pd.DataFrame({
        'date_time': X_val.index,
        'actual': y_val,
        'predicted': y_val_pred
    })
    val_predictions_path = os.path.join(predictions_dir, f'{model_name.lower()}_val_predictions.csv')
    val_predictions_df.to_csv(val_predictions_path, index=False)
    print(f"{model_name} validation predictions saved: {val_predictions_path}")

    # Test set prediction and evaluation
    y_test_pred = model.predict(X_test_scaled)
    test_results = evaluate_model(y_test, y_test_pred, model_name, "Test")
    results.append(test_results)
    # Save test predictions
    test_predictions_df = pd.DataFrame({
        'date_time': X_test.index,
        'actual': y_test,
        'predicted': y_test_pred
    })
    test_predictions_path = os.path.join(predictions_dir, f'{model_name.lower()}_test_predictions.csv')
    test_predictions_df.to_csv(test_predictions_path, index=False)
    print(f"{model_name} test predictions saved: {test_predictions_path}")

    # Save model
    model_path = os.path.join(models_dir, f'{model_name.lower()}_model.pkl')
    joblib.dump(model, model_path)
    print(f"{model_name} model saved: {model_path}")

# Compile and save results
results_df = pd.DataFrame(results)
results_df.to_csv(os.path.join(base_dir, 'model_results.csv'), index=False)
print("\nAll Results:\n", results_df)
print("\nModel evaluation and prediction saving completed.")

In [ ]:
X_train_scaled.max().max(), X_train_scaled.min().min()

In [ ]:
X_val_scaled.max().max(), X_val_scaled.min().min()

In [ ]:
X_test_scaled.max().max(), X_test_scaled.min().min()

In [ ]:
X_test_scaled.min().sort_values()

In [ ]:
def evaluate_and_save_predictions(y_true, y_pred, model_name, X, dataset, predictions_dir):
    # Ensure y_true and y_pred have the same length for evaluation
    if len(y_true) != len(y_pred):
        # This should not happen if prediction_generator is correct, but as a safeguard
        min_len = min(len(y_true), len(y_pred))
        y_true = y_true[:min_len]
        y_pred = y_pred[:min_len]
        print(f"Warning: Length mismatch between y_true and y_pred for {model_name} ({dataset}). Truncating to {min_len}.")

    results = evaluate_model(y_true, y_pred, model_name, dataset)

    # Save predictions to CSV
    # Adjust slicing for Persistence model predictions
    if model_name == "Persistence":
        # Use the already sliced X_subset from evaluate_baseline_model
        predictions_df = pd.DataFrame({
            'date_time': X.index,
            'actual': y_true, # y_true is already sliced in persistence_prediction
            'predicted': y_pred
        })
    else:
         predictions_df = pd.DataFrame({
            'date_time': X.index,
            'actual': y_true,
            'predicted': y_pred
        })

    filename = f"{model_name.lower().replace(' ', '_').replace('(', '').replace(')', '')}_{dataset.lower()}_predictions.csv"
    predictions_path = os.path.join(predictions_dir, filename)
    predictions_df.to_csv(predictions_path, index=False)
    print(f"{model_name} {dataset} predictions saved: {predictions_path}")

    return results

def evaluate_baseline_model(y_true_dict, X_dict, model_name, prediction_generator, predictions_dir):
    results = []
    for dataset, y_true, X in zip(y_true_dict.keys(), y_true_dict.values(), X_dict.values()):
        y_pred = prediction_generator(X, y_true, dataset)
        # Pass the appropriate y_true subset to evaluate_and_save_predictions
        if model_name == "Persistence":
            y_true_subset = y_true[1:]
            X_subset = X.iloc[1:] # Also slice X for correct index alignment
        else:
            y_true_subset = y_true
            X_subset = X
        results.append(evaluate_and_save_predictions(y_true_subset, y_pred, model_name, X_subset, dataset, predictions_dir))
    return results

In [ ]:
# Directory setup
base_dir = '/content/drive/MyDrive/windforecast/'
predictions_dir = os.path.join(base_dir, 'predictions')
os.makedirs(predictions_dir, exist_ok=True)
print(f"Predictions directory: {predictions_dir}")

# Ensure that necessary columns are added
if 'year' not in df.columns:
    df['year'] = df.index.year
if 'day_of_year' not in df.columns:
    df['day_of_year'] = df.index.dayofyear

# Data for baseline models
y_true_dict = {'Train': y_train, 'Validation': y_val, 'Test': y_test}
X_dict = {'Train': X_train, 'Validation': X_val, 'Test': X_test}

# Baseline Models Evaluation
baseline_results = []

# Persistence Baseline
def persistence_prediction(X, y_true, dataset):
    # Return predictions aligned with y_true[1:]
    y_pred = y_true.shift(1).fillna(y_true.mean())
    return y_pred[1:]

results_persistence = evaluate_baseline_model(y_true_dict, X_dict, "Persistence",
    lambda X, y_true, dataset: persistence_prediction(X, y_true, dataset), predictions_dir)
baseline_results.extend(results_persistence)

# Seasonal-Naive Baseline
df['seasonal_naive_pred'] = df.groupby(['day_of_year'])['target'].transform('mean')
def seasonal_naive_prediction(X, y_true, dataset):
    return df.loc[X.index, 'seasonal_naive_pred']

results_seasonal = evaluate_baseline_model(y_true_dict, X_dict, "Seasonal-Naive (Mean)",
    seasonal_naive_prediction, predictions_dir)
baseline_results.extend(results_seasonal)

# Moving Average (7 days)
window = 7
df['moving_avg_pred'] = df['target'].rolling(window=window, min_periods=1).mean().shift(1)
def moving_avg_prediction(X, y_true, dataset):
    return df.loc[X.index, 'moving_avg_pred'].fillna(y_true.mean())

results_moving_avg = evaluate_baseline_model(y_true_dict, X_dict, "Moving Average (7-day)",
    moving_avg_prediction, predictions_dir)
baseline_results.extend(results_moving_avg)

# Combine and save all results
results_df_new = pd.DataFrame(baseline_results)
try:
    existing_results = pd.read_csv(os.path.join(base_dir, 'model_results.csv'))
    results_df = pd.concat([existing_results, results_df_new], ignore_index=True)
except FileNotFoundError:
    print("Existing results file not found, saving only the current results.")
    results_df = results_df_new

results_df.to_csv(os.path.join(base_dir, 'model_results.csv'), index=False)
print("\nAll Results:\n", results_df)
print("\nBaseline evaluation and prediction saving completed.")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LassoCV, ElasticNetCV
from statsmodels.stats.outliers_influence import variance_inflation_factor
import os
import joblib

def max_absolute_error(y_true, y_pred):
    return np.max(np.abs(y_true - y_pred))

def evaluate_model(y_true, y_pred, model_name, dataset="Validation"):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    max_ae = max_absolute_error(y_true, y_pred)
    print(f"\n{model_name} Performance ({dataset} Set):")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"R²: {r2:.4f}")
    print(f"MaxAE: {max_ae:.4f}")
    return {'Model': model_name, 'Dataset': dataset, 'RMSE': rmse, 'MAE': mae, 'R²': r2, 'MaxAE': max_ae}

def train_and_evaluate_model(model, model_name, X_train, y_train, X_val, y_val, X_test, y_test, predictions_dir, models_dir):
    try:
        # Train the model
        model.fit(X_train, y_train)

        # Evaluate on train, validation, and test datasets
        datasets = [('Train', X_train, y_train),
                    ('Validation', X_val, y_val),
                    ('Test', X_test, y_test)]

        model_results = []

        for dataset_name, X, y in datasets:
            y_pred = model.predict(X)
            results = evaluate_model(y, y_pred, f"{model_name}_Selected", dataset_name)
            model_results.append(results)

            # Save predictions
            predictions_df = pd.DataFrame({
                'date_time': X.index,
                'actual': y,
                'predicted': y_pred
            })
            filename = f"{model_name.lower()}_selected_simplified_{dataset_name.lower()}_predictions.csv"
            predictions_path = os.path.join(predictions_dir, filename)
            predictions_df.to_csv(predictions_path, index=False)
            print(f"{model_name}_Selected {dataset_name.lower()} predictions saved: {predictions_path}")

        # Save the model
        model_path = os.path.join(models_dir, f'{model_name.lower()}_selected_simplified_model.pkl')
        joblib.dump(model, model_path)
        print(f"{model_name}_Selected model saved: {model_path}")

        return model_results

    except Exception as e:
        print(f"Error in {model_name} model: {str(e)}")
        return None

# Directory setup
base_dir = '/content/drive/MyDrive/windforecast/'
predictions_dir = os.path.join(base_dir, 'predictions')
models_dir = os.path.join(base_dir, 'models')
os.makedirs(predictions_dir, exist_ok=True)
os.makedirs(models_dir, exist_ok=True)
print(f"Predictions directory: {predictions_dir}")
print(f"Models directory: {models_dir}")

# NaN kontrolü
print("X_train_scaled NaN sayısı:", X_train_scaled.isna().sum().sum())
print("X_val_scaled NaN sayısı:", X_val_scaled.isna().sum().sum())
print("X_test_scaled NaN sayısı:", X_test_scaled.isna().sum().sum())
assert X_train_scaled.isna().sum().sum() == 0, "X_train_scaled'da NaN var!"
assert X_val_scaled.isna().sum().sum() == 0, "X_val_scaled'da NaN var!"
assert X_test_scaled.isna().sum().sum() == 0, "X_test_scaled'da NaN var!"

# Shape kontrolü
print("X_train_scaled shape:", X_train_scaled.shape)
print("X_val_scaled shape:", X_val_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)
assert X_train_scaled.shape[1] == X_val_scaled.shape[1] == X_test_scaled.shape[1], "Feature sayıları uyumsuz!"

# Random Forest ile feature importance
rf = RandomForestRegressor(n_estimators=1000, random_state=42, n_jobs=-1)
rf.fit(X_train_scaled, y_train)

# Feature importance
importances = pd.DataFrame({'feature': X_train_scaled.columns, 'importance': rf.feature_importances_})
importances = importances.sort_values(by='importance', ascending=False)
print("\nTop 10 feature:\n", importances.head(10))
print(f"Toplam feature sayısı: {len(importances)}")
print(f"Önemi < 0.005 olan feature sayısı: {len(importances[importances['importance'] < 0.005])}")

# Önemli feature'ları seç (threshold: 0.005)
selected_features = importances[importances['importance'] >= 0.005]['feature'].tolist()
print(f"Seçilen feature sayısı: {len(selected_features)}")

# VIF ile multicollinearity kontrolü
def calculate_vif(X, max_features=100):
    X_subset = X.iloc[:, :min(max_features, X.shape[1])]
    vif_data = pd.DataFrame()
    vif_data['feature'] = X_subset.columns
    try:
        vif_data['VIF'] = [variance_inflation_factor(X_subset.values, i) for i in range(X_subset.shape[1])]
    except:
        print("VIF hesaplama hatası, multicollinearity kontrolü atlanıyor.")
        return pd.DataFrame({'feature': X_subset.columns, 'VIF': [1] * X_subset.shape[1]})
    return vif_data

vif = calculate_vif(X_train_scaled[selected_features])
print("\nYüksek VIF (>10):\n", vif[vif['VIF'] > 10])

# Yüksek VIF'li feature'ları çıkar
low_vif_features = vif[vif['VIF'] <= 10]['feature'].tolist()
if not low_vif_features:
    low_vif_features = selected_features
    print("VIF <= 10 feature bulunamadı, tüm seçilen feature'lar kullanılıyor.")
print(f"Son seçilen feature sayısı (VIF <= 10): {len(low_vif_features)}")

# Feature importance ve VIF sonuçlarını kaydet
importances.to_csv(os.path.join(base_dir, 'feature_importances_simplified.csv'), index=False)
vif.to_csv(os.path.join(base_dir, 'vif_results_simplified.csv'), index=False)
print(f"Feature importance kaydedildi: {os.path.join(base_dir, 'feature_importances_simplified.csv')}")
print(f"VIF sonuçları kaydedildi: {os.path.join(base_dir, 'vif_results_simplified.csv')}")

# Seçilen feature'larla veri setlerini güncelle
X_train_selected = X_train_scaled[low_vif_features]
X_val_selected = X_val_scaled[low_vif_features]
X_test_selected = X_test_scaled[low_vif_features]

print("Seçilen feature'larla veri setleri güncellendi:")
print("X_train_selected shape:", X_train_selected.shape)
print("X_val_selected shape:", X_val_selected.shape)
print("X_test_selected shape:", X_test_selected.shape)

# Define models
alphas = [0.001, 0.003, 0.005, 0.007, 0.01, 0.03, 0.05, 0.07, 0.1, 0.3, 0.5, 0.7, 1, 3, 5, 7, 10, 15, 20, 25, 30, 50, 100]
l1_ratios = [0.1, 0.3, 0.5, 0.7, 0.9]

models = {
    'LassoCV': LassoCV(cv=5, alphas=alphas, random_state=42, max_iter=200000),
    'ElasticNetCV': ElasticNetCV(cv=5, l1_ratio=l1_ratios, alphas=alphas, random_state=42, max_iter=200000),
    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
}

# Training and evaluation
results = []
for model_name, model in models.items():
    print(f"\nEvaluating {model_name} (Selected Features)...")
    model_results = train_and_evaluate_model(model, model_name,
                                            X_train_selected, y_train,
                                            X_val_selected, y_val,
                                            X_test_selected, y_test,
                                            predictions_dir, models_dir)
    if model_results:
        results.extend(model_results)

# Combine and save all results
results_df_new = pd.DataFrame(results)
try:
    existing_results = pd.read_csv(os.path.join(base_dir, 'model_results.csv'))
    results_df = pd.concat([existing_results, results_df_new], ignore_index=True)
except FileNotFoundError:
    print("Existing results file not found, saving only the current results.")
    results_df = results_df_new

results_df.to_csv(os.path.join(base_dir, 'model_results.csv'), index=False)
print("\nAll Results:\n", results_df)
print("\nFeature selection and model evaluation completed.")

# XGBOOST - First Trial

In [ ]:
import xgboost as xgb
!pip install optuna
import optuna
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import pandas as pd
import numpy as np

# Assuming X_train, y_train, X_val, y_val, X_test, y_test are already defined and preprocessed
# Also assuming that feature selection has been applied if needed

def objective(trial):
    params = {
        'tree_method': 'gpu_hist',  # Use GPU acceleration
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1),
        'gamma': trial.suggest_float('gamma', 0, 0.5),
        'lambda': trial.suggest_float('lambda', 0, 1.0),
        'alpha': trial.suggest_float('alpha', 0, 1.0),
        'objective': 'reg:squarederror',
        'seed': 42,
        'verbosity': 0
    }

    model = xgb.XGBRegressor(**params)

    # TimeSeriesSplit for time series cross-validation
    tscv = TimeSeriesSplit(n_splits=5)
    rmse_scores = []

    for train_index, val_index in tscv.split(X_train):
        X_train_cv, X_val_cv = X_train.iloc[train_index], X_train.iloc[val_index]
        y_train_cv, y_val_cv = y_train.iloc[train_index], y_train.iloc[val_index]

        model.fit(X_train_cv, y_train_cv)
        y_pred = model.predict(X_val_cv)
        rmse = np.sqrt(mean_squared_error(y_val_cv, y_pred))
        rmse_scores.append(rmse)

    return np.mean(rmse_scores)

# Optuna study
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

# Get the best hyperparameters
best_params = study.best_params
print("Best hyperparameters: ", best_params)

# Train the final model with the best hyperparameters
best_xgb_model = xgb.XGBRegressor(**best_params)
best_xgb_model.fit(X_train, y_train)

# Evaluate on validation and test sets
def evaluate_model(y_true, y_pred, dataset_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"\nPerformance on {dataset_name} set:")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"R²: {r2:.4f}")

# Validation set predictions
y_val_pred = best_xgb_model.predict(X_val)
evaluate_model(y_val, y_val_pred, "Validation")

# Test set predictions
y_test_pred = best_xgb_model.predict(X_test)
evaluate_model(y_test, y_test_pred, "Test")

# Save the model
model_path = '/content/drive/MyDrive/windforecast/models/xgboost_hpo_model.pkl'
joblib.dump(best_xgb_model, model_path)
print(f"Model saved to {model_path}")

# Feature importance
importances = pd.DataFrame({
    'feature': X_train.columns,
    'importance': best_xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 features:\n", importances.head(10))

# Save feature importance
importances.to_csv('/content/drive/MyDrive/windforecast/xgboost_feature_importance.csv', index=False)
print("Feature importance saved to CSV.")

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import pandas as pd
import numpy as np
import joblib
import os

def max_absolute_error(y_true, y_pred):
    return np.max(np.abs(y_true - y_pred))

def evaluate_model(y_true, y_pred, model_name, dataset_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    max_ae = max_absolute_error(y_true, y_pred)

    print(f"\nPerformance on {dataset_name} set:")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"R²: {r2:.4f}")
    print(f"MaxAE: {max_ae:.4f}")

    return {'Model': model_name, 'Dataset': dataset_name, 'RMSE': rmse, 'MAE': mae, 'R²': r2, 'MaxAE': max_ae}

# Assume X_train_scaled, y_train, X_val_scaled, y_val, X_test_scaled, y_test are already defined

base_dir = '/content/drive/MyDrive/windforecast/'
os.makedirs(os.path.join(base_dir, 'models'), exist_ok=True)

# Get the best hyperparameters
best_params = study.best_params
print("Best hyperparameters:", best_params)

# Train the final model with the best hyperparameters on the full training data
best_xgb_model = xgb.XGBRegressor(**best_params)
best_xgb_model.fit(X_train, y_train)

# Evaluate on the training set
y_train_pred = best_xgb_model.predict(X_train)
train_results = evaluate_model(y_train, y_train_pred, "XGBoost", "Train")

# Evaluate on validation set
y_val_pred = best_xgb_model.predict(X_val)
validation_results = evaluate_model(y_val, y_val_pred, "XGBoost", "Validation")

# # Train the model once again on combined train and validation data

# Evaluate on test set using the CV-trained model
y_test_pred = best_xgb_model.predict(X_test)
test_results = evaluate_model(y_test, y_test_pred, "XGBoost", "Test")

# Save the model
model_path = os.path.join(base_dir, 'models/xgboost_hpo_model.pkl')
joblib.dump(best_xgb_model, model_path)
print(f"Model saved to {model_path}")

# Feature importance
importances = pd.DataFrame({
    'feature': X_train_scaled.columns,
    'importance': best_xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 features:\n", importances.head(10))

# Save feature importance
importances_path = os.path.join(base_dir, 'xgboost_feature_importance.csv')
importances.to_csv(importances_path, index=False)
print(f"Feature importance saved to {importances_path}")

# Prepare results dictionaries for DataFrame update
xgboost_train_results = train_results
xgboost_validation_results = validation_results
xgboost_test_results = test_results

# Combine new results into a DataFrame
new_results_df = pd.DataFrame([xgboost_train_results, xgboost_validation_results, xgboost_test_results])

# Path to the results file
results_path = os.path.join(base_dir, 'model_results.csv')

# Load existing results if the file exists
try:
    existing_results_df = pd.read_csv(results_path)
    updated_results_df = pd.concat([existing_results_df, new_results_df], ignore_index=True)
except FileNotFoundError:
    updated_results_df = new_results_df
    print("Existing results file not found, saving current results only.")

# Save the updated results DataFrame to CSV
updated_results_df.to_csv(results_path, index=False)
print("XGBoost results added to the results DataFrame and saved to CSV.")

# Display the updated DataFrame
print("\nUpdated Results DataFrame:\n", updated_results_df)

In [ ]:
import lightgbm as lgb
import optuna
from sklearn.model_selection import TimeSeriesSplit
import pandas as pd
import numpy as np
import joblib
import os

def max_absolute_error(y_true, y_pred):
    return np.max(np.abs(y_true - y_pred))

def evaluate_model(y_true, y_pred, model_name, dataset_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    max_ae = max_absolute_error(y_true, y_pred)
    print(f"\nPerformance on {dataset_name} set:")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"R²: {r2:.4f}")
    print(f"MaxAE: {max_ae:.4f}")
    return {'Model': model_name, 'Dataset': dataset_name, 'RMSE': rmse, 'MAE': mae, 'R²': r2, 'MaxAE': max_ae}

# Assume X_train, y_train, X_val, y_val, X_test, y_test are already defined
base_dir = '/content/drive/MyDrive/windforecast/'
os.makedirs(os.path.join(base_dir, 'models'), exist_ok=True)

def objective(trial):
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 1),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 1.0),
        'random_state': 42,
        'verbosity': -1,
        'device': 'gpu'  # Use GPU if available
    }

    model = lgb.LGBMRegressor(**params)

    tscv = TimeSeriesSplit(n_splits=5)
    rmse_scores = []

    for train_index, val_index in tscv.split(X_train):
        X_train_cv, X_val_cv = X_train.iloc[train_index], X_train.iloc[val_index]
        y_train_cv, y_val_cv = y_train.iloc[train_index], y_train.iloc[val_index]

        model.fit(X_train_cv, y_train_cv)
        y_pred = model.predict(X_val_cv)
        rmse = np.sqrt(mean_squared_error(y_val_cv, y_pred))
        rmse_scores.append(rmse)

    return np.mean(rmse_scores)

# Optuna study setup
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

# Retrieve the best hyperparameters
best_params = study.best_params
print("Best hyperparameters:", best_params)

# Train the final model with the best hyperparameters on the full training data
best_lgbm_model = lgb.LGBMRegressor(**best_params)
best_lgbm_model.fit(X_train, y_train)

# Evaluate on the training set
y_train_pred = best_lgbm_model.predict(X_train)
train_results = evaluate_model(y_train, y_train_pred, "LightGBM", "Train")

# Evaluate on validation set
y_val_pred = best_lgbm_model.predict(X_val)
validation_results = evaluate_model(y_val, y_val_pred, "LightGBM", "Validation")

# Evaluate on test set
y_test_pred = best_lgbm_model.predict(X_test)
test_results = evaluate_model(y_test, y_test_pred, "LightGBM", "Test")

# Save the model
model_path = os.path.join(base_dir, 'models/lightgbm_hpo_model.pkl')
joblib.dump(best_lgbm_model, model_path)
print(f"Model saved to {model_path}")

# Feature importance
importances = pd.DataFrame({
    'feature': X_train.columns,
    'importance': best_lgbm_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 features:\n", importances.head(10))

# Save feature importance
importances_path = os.path.join(base_dir, 'lightgbm_feature_importance.csv')
importances.to_csv(importances_path, index=False)
print(f"Feature importance saved to {importances_path}")

# Prepare results dictionaries for DataFrame update
lgbm_train_results = train_results
lgbm_validation_results = validation_results
lgbm_test_results = test_results

# Combine new results into a DataFrame
new_results_df = pd.DataFrame([lgbm_train_results, lgbm_validation_results, lgbm_test_results])

# Path to the results file
results_path = os.path.join(base_dir, 'model_results.csv')

# Load existing results if the file exists
try:
    existing_results_df = pd.read_csv(results_path)
    updated_results_df = pd.concat([existing_results_df, new_results_df], ignore_index=True)
except FileNotFoundError:
    updated_results_df = new_results_df
    print("Existing results file not found, saving current results only.")

# Save the updated results DataFrame to CSV
updated_results_df.to_csv(results_path, index=False)
print("LightGBM results added to the results DataFrame and saved to CSV.")

# Display the updated DataFrame
print("\nUpdated Results DataFrame:\n", updated_results_df)

In [ ]:
!pip install catboost
import catboost as cb

def max_absolute_error(y_true, y_pred):
    return np.max(np.abs(y_true - y_pred))

def evaluate_model(y_true, y_pred, model_name, dataset_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    max_ae = max_absolute_error(y_true, y_pred)
    print(f"\nPerformance on {dataset_name} set:")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"R²: {r2:.4f}")
    print(f"MaxAE: {max_ae:.4f}")
    return {'Model': model_name, 'Dataset': dataset_name, 'RMSE': rmse, 'MAE': mae, 'R²': r2, 'MaxAE': max_ae}

# Assume X_train, y_train, X_val, y_val, X_test, y_test are already defined
base_dir = '/content/drive/MyDrive/windforecast/'
os.makedirs(os.path.join(base_dir, 'models'), exist_ok=True)

def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 50, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-8, 10.0, log=True),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'random_state': 42,
        'verbose': False,
        'task_type': 'GPU'  # Use GPU if available
    }

    model = cb.CatBoostRegressor(**params)

    tscv = TimeSeriesSplit(n_splits=5)
    rmse_scores = []

    for train_index, val_index in tscv.split(X_train):
        X_train_cv, X_val_cv = X_train.iloc[train_index], X_train.iloc[val_index]
        y_train_cv, y_val_cv = y_train.iloc[train_index], y_train.iloc[val_index]

        model.fit(X_train_cv, y_train_cv, verbose=False)
        y_pred = model.predict(X_val_cv)
        rmse = np.sqrt(mean_squared_error(y_val_cv, y_pred))
        rmse_scores.append(rmse)

    return np.mean(rmse_scores)

# Optuna study setup
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

# Retrieve the best hyperparameters
best_params = study.best_params
print("Best hyperparameters:", best_params)

# Train the final model with the best hyperparameters on the full training data
best_catboost_model = cb.CatBoostRegressor(**best_params, verbose=False)
best_catboost_model.fit(X_train, y_train)

# Evaluate on the training set
y_train_pred = best_catboost_model.predict(X_train)
train_results = evaluate_model(y_train, y_train_pred, "CatBoost", "Train")

# Evaluate on validation set
y_val_pred = best_catboost_model.predict(X_val)
validation_results = evaluate_model(y_val, y_val_pred, "CatBoost", "Validation")

# Evaluate on test set
y_test_pred = best_catboost_model.predict(X_test)
test_results = evaluate_model(y_test, y_test_pred, "CatBoost", "Test")

# Save the model
model_path = os.path.join(base_dir, 'models/catboost_hpo_model.pkl')
joblib.dump(best_catboost_model, model_path)
print(f"Model saved to {model_path}")

# Feature importance
importances = pd.DataFrame({
    'feature': X_train.columns,
    'importance': best_catboost_model.get_feature_importance()
}).sort_values('importance', ascending=False)

print("\nTop 10 features:\n", importances.head(10))

# Save feature importance
importances_path = os.path.join(base_dir, 'catboost_feature_importance.csv')
importances.to_csv(importances_path, index=False)
print(f"Feature importance saved to {importances_path}")

# Prepare results dictionaries for DataFrame update
catboost_train_results = train_results
catboost_validation_results = validation_results
catboost_test_results = test_results

# Combine new results into a DataFrame
new_results_df = pd.DataFrame([catboost_train_results, catboost_validation_results, catboost_test_results])

# Path to the results file
results_path = os.path.join(base_dir, 'model_results.csv')

# Load existing results if the file exists
try:
    existing_results_df = pd.read_csv(results_path)
    updated_results_df = pd.concat([existing_results_df, new_results_df], ignore_index=True)
except FileNotFoundError:
    updated_results_df = new_results_df
    print("Existing results file not found, saving current results only.")

# Save the updated results DataFrame to CSV
updated_results_df.to_csv(results_path, index=False)
print("CatBoost results added to the results DataFrame and saved to CSV.")

# Display the updated DataFrame
print("\nUpdated Results DataFrame:\n", updated_results_df)

In [ ]:
%load_ext cuml.accel
from sklearn.ensemble import RandomForestRegressor
!pip install optuna
import optuna
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import pandas as pd
import numpy as np
import joblib
import os

def max_absolute_error(y_true, y_pred):
    return np.max(np.abs(y_true - y_pred))

def evaluate_model(y_true, y_pred, model_name, dataset_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    max_ae = max_absolute_error(y_true, y_pred)
    print(f"\nPerformance on {dataset_name} set:")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"R²: {r2:.4f}")
    print(f"MaxAE: {max_ae:.4f}")
    return {'Model': model_name, 'Dataset': dataset_name, 'RMSE': rmse, 'MAE': mae, 'R²': r2, 'MaxAE': max_ae}

# Assume X_train, y_train, X_val, y_val, X_test, y_test are already defined
base_dir = '/content/drive/MyDrive/windforecast/'
os.makedirs(os.path.join(base_dir, 'models'), exist_ok=True)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 20, 3000),
        'max_depth': trial.suggest_int('max_depth', 2, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 30),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 30),
        'max_features': trial.suggest_float('max_features', 0.5, 1.0, step=0.05),
        'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),
        'random_state': 42
    }

    model = RandomForestRegressor(**params)

    tscv = TimeSeriesSplit(n_splits=5)
    rmse_scores = []

    for train_index, val_index in tscv.split(X_train):
        X_train_cv, X_val_cv = X_train.iloc[train_index], X_train.iloc[val_index]
        y_train_cv, y_val_cv = y_train.iloc[train_index], y_train.iloc[val_index]

        model.fit(X_train_cv, y_train_cv)
        y_pred = model.predict(X_val_cv)
        rmse = np.sqrt(mean_squared_error(y_val_cv, y_pred))
        rmse_scores.append(rmse)

    return np.mean(rmse_scores)

# Optuna study setup
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

In [ ]:
# Retrieve the best hyperparameters

best_params = {'n_estimators': 942, 'max_depth': 19, 'min_samples_split': 12, 'min_samples_leaf': 5, 'max_features': 1.0, 'bootstrap': True} # study.best_params
print("Best hyperparameters:", best_params)

# unload cuml.accel *(%load_ext cuml.accel)
%unload_ext cuml.accel

from sklearn.ensemble import RandomForestRegressor

# Train the final model with the best hyperparameters on the full training data
best_rf_model = RandomForestRegressor(**best_params)
best_rf_model.fit(X_train, y_train)

# Evaluate on the training set
y_train_pred = best_rf_model.predict(X_train)
train_results = evaluate_model(y_train, y_train_pred, "RandomForest", "Train")

# Evaluate on validation set
y_val_pred = best_rf_model.predict(X_val)
validation_results = evaluate_model(y_val, y_val_pred, "RandomForest", "Validation")

# Evaluate on test set
y_test_pred = best_rf_model.predict(X_test)
test_results = evaluate_model(y_test, y_test_pred, "RandomForest", "Test")

# Save the model
!pip install skops
import skops.io as sio
base_dir = '/content/drive/MyDrive/windforecast/'
os.makedirs(os.path.join(base_dir, 'models'), exist_ok=True)
model_path = os.path.join(base_dir, 'models/random_forest_hpo_model.skops')
obj = sio.dump(best_rf_model, model_path)
print(f"Model saved to {model_path}")

# Feature importance
importances = pd.DataFrame({
    'feature': X_train.columns,
    'importance': best_rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 features:\n", importances.head(10))

# Save feature importance
importances_path = os.path.join(base_dir, 'randomforest_feature_importance.csv')
importances.to_csv(importances_path, index=False)
print(f"Feature importance saved to {importances_path}")

# Prepare results dictionaries for DataFrame update
rf_train_results = train_results
rf_validation_results = validation_results
rf_test_results = test_results

# Combine new results into a DataFrame
new_results_df = pd.DataFrame([rf_train_results, rf_validation_results, rf_test_results])

# Path to the results file
results_path = os.path.join(base_dir, 'model_results.csv')

# Load existing results if the file exists
try:
    existing_results_df = pd.read_csv(results_path)
    updated_results_df = pd.concat([existing_results_df, new_results_df], ignore_index=True)
except FileNotFoundError:
    updated_results_df = new_results_df
    print("Existing results file not found, saving current results only.")

# Save the updated results DataFrame to CSV
updated_results_df.to_csv(results_path, index=False)
print("Random Forest results added to the results DataFrame and saved to CSV.")

# Display the updated DataFrame
print("\nUpdated Results DataFrame:\n", updated_results_df)

In [ ]:
# Permutation feature importances (Random Forest) via eli5 -> Table 5
!pip install eli5
import eli5
from eli5.sklearn import PermutationImportance

perm = PermutationImportance(best_rf_model, random_state=42).fit(X_train.values, y_train.values)
eli5.show_weights(perm, feature_names=X_train.columns.tolist())

In [ ]:
def objective_svr(trial):
    kernel = trial.suggest_categorical('kernel', ['poly', 'rbf'])
    params = {
        'kernel': kernel,
        'C': trial.suggest_float('C', 0.01, 200, log=True),  # Expanded range
        'epsilon': trial.suggest_float('epsilon', 0.001, 15, log=True),  # Expanded range
        'gamma': trial.suggest_categorical('gamma', ['scale', 'auto'])
    }

    if params['kernel'] == 'poly':
        params['degree'] = trial.suggest_int('degree', 2, 7)  # Expanded range

    model = SVR(**params)

    tscv = TimeSeriesSplit(n_splits=5)
    rmse_scores = []

    for train_index, val_index in tscv.split(X_train_scaled):
        X_train_cv, X_val_cv = X_train_scaled[train_index], X_train_scaled[val_index]
        y_train_cv, y_val_cv = y_train.iloc[train_index], y_train.iloc[val_index]

        model.fit(X_train_cv, y_train_cv)
        y_pred = model.predict(X_val_cv)
        rmse = np.sqrt(mean_squared_error(y_val_cv, y_pred))
        rmse_scores.append(rmse)

    return np.mean(rmse_scores)

# Optuna study setup for SVR with non-linear kernels
study_svr = optuna.create_study(direction='minimize')
study_svr.optimize(objective_svr, n_trials=50)

# Retrieve the best hyperparameters
best_params_svr = study_svr.best_params
print("Best hyperparameters for SVR:", best_params_svr)

# Train the final SVR model with the best hyperparameters on the full training data
best_svr_model = SVR(**best_params_svr)
best_svr_model.fit(X_train_scaled, y_train)

# Evaluate on the training set
y_train_pred = best_svr_model.predict(X_train_scaled)
train_results_svr = evaluate_model(y_train, y_train_pred, "SVR", "Train")

# Evaluate on validation set
y_val_pred = best_svr_model.predict(X_val_scaled)
validation_results_svr = evaluate_model(y_val, y_val_pred, "SVR", "Validation")

# Evaluate on test set
y_test_pred = best_svr_model.predict(X_test_scaled)
test_results_svr = evaluate_model(y_test, y_test_pred, "SVR", "Test")

# Save the model
model_path_svr = os.path.join(base_dir, 'models/svr_hpo_model.pkl')
joblib.dump(best_svr_model, model_path_svr)
print(f"SVR Model saved to {model_path_svr}")

# Prepare results dictionaries for DataFrame update for SVR
svr_train_results = train_results_svr
svr_validation_results = validation_results_svr
svr_test_results = test_results_svr

# Combine new results into a DataFrame
new_results_df = pd.DataFrame([
    svr_train_results, svr_validation_results, svr_test_results
])

# Path to the results file
results_path = os.path.join(base_dir, 'model_results.csv')

# Load existing results if the file exists
try:
    existing_results_df = pd.read_csv(results_path)
    updated_results_df = pd.concat([existing_results_df, new_results_df], ignore_index=True)
except FileNotFoundError:
    updated_results_df = new_results_df
    print("Existing results file not found, saving current results only.")

# Save the updated results DataFrame to CSV
updated_results_df.to_csv(results_path, index=False)
print("Results added to the results DataFrame and saved to CSV.")

# Display the updated DataFrame
print("\nUpdated Results DataFrame:\n", updated_results_df)

In [ ]:
!pip install optuna
import optuna
import xgboost as xgb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

# Assuming your datasets are already prepared and available as X_train, y_train, X_val, y_val, X_test, y_test

def objective(trial):
    # Define the hyperparameter search space
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 20, 2000),
        'max_depth': trial.suggest_int('max_depth', 3, 16),
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.3, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 1),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 1),
        "device": "cuda",
        "tree_method": "hist"
    }

    model = xgb.XGBRegressor(**params, random_state=42)

    # TimeSeries cross-validation
    tscv = TimeSeriesSplit(n_splits=5)
    rmses = []

    for train_idx, val_idx in tscv.split(X_train):
        X_train_cv, X_val_cv = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_train_cv, y_val_cv = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model.fit(X_train_cv, y_train_cv)
        preds = model.predict(X_val_cv)
        rmse = np.sqrt(mean_squared_error(y_val_cv, preds))
        rmses.append(rmse)

    return np.mean(rmses)

def evaluate_and_print(y_true, y_pred, dataset_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"Performance on {dataset_name} set:")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"R²: {r2:.4f}")
    return rmse, mae, r2

# Optuna setup with pruning
study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler())
study.optimize(objective, n_trials=400)

# Retrieve best parameters and train the final model
best_params = study.best_params
final_model = xgb.XGBRegressor(**best_params, random_state=42)
final_model.fit(X_train, y_train)

# Evaluate on the validation and test sets
y_val_pred = final_model.predict(X_val)
y_test_pred = final_model.predict(X_test)

val_metrics = evaluate_and_print(y_val, y_val_pred, "Validation")
test_metrics = evaluate_and_print(y_test, y_test_pred, "Test")

# Plot feature importance
fig, ax = plt.subplots(figsize=(10, 8))
xgb.plot_importance(final_model, ax=ax)
plt.title('XGBoost Feature Importance')
plt.show()

# Plot learning curves (requires evals_result)
# This requires you to pass eval_set during training if you're logging training progress
# Here's a simplified approach using training data
eval_set = [(X_train, y_train), (X_val, y_val)]
final_model.fit(X_train, y_train, eval_set=eval_set, eval_metric='rmse', verbose=False)

results = final_model.evals_result()
epochs = len(results['validation_0']['rmse'])
x_axis = range(0, epochs)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(x_axis, results['validation_0']['rmse'], label='Train')
ax.plot(x_axis, results['validation_1']['rmse'], label='Validation')
ax.legend()
plt.ylabel('RMSE')
plt.title('XGBoost Learning Curves')
plt.show()

In [ ]:
import xgboost as xgb
import numpy as np
# Retrieve best parameters and train the final model
best_params = {'n_estimators': 1834, 'max_depth': 11, 'learning_rate': 0.004854856793781717, 'subsample': 0.40566526847932777, 'colsample_bytree': 0.3968091211824199, 'gamma': 1.4684181309759596, 'min_child_weight': 2, 'reg_alpha': 0.9001406603742224, 'reg_lambda': 0.23250578488239765}
print("Best hyperparameters:", best_params)
final_model = xgb.XGBRegressor(**best_params, random_state=42)
eval_set = [(X_train, y_train), (X_val, y_val)]
final_model.fit(X_train, y_train, eval_set=eval_set, verbose=False)
results = final_model.evals_result()

# Evaluate on the validation and test sets
y_val_pred = final_model.predict(X_val)
y_test_pred = final_model.predict(X_test)

val_metrics = evaluate_model(y_val, y_val_pred, "XGBoost-HPO", "Validation")
test_metrics = evaluate_model(y_test, y_test_pred, "XGBoost-HPO", "Test")

# Plot feature importance
fig, ax = plt.subplots(figsize=(10, 8))
xgb.plot_importance(final_model, ax=ax)
plt.title('XGBoost Feature Importance')
plt.show()

# Plot learning curves (requires evals_result)
# This requires you to pass eval_set during training if you're logging training progress
# Here's a simplified approach using training data

epochs = len(results['validation_0']['rmse'])
x_axis = range(0, epochs)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(x_axis, results['validation_0']['rmse'], label='Train')
ax.plot(x_axis, results['validation_1']['rmse'], label='Validation')
ax.legend()
plt.ylabel('RMSE')
plt.title('XGBoost Learning Curves')
plt.show()

In [ ]:
epochs

In [ ]:
results

In [ ]:
models_dir

In [ ]:
final_model.save_model(models_dir + '/best_xgboost_model.json')

import joblib

# Assuming 'final_model' is your trained XGBoost model
model_filename = models_dir + '/best_xgboost_model.pkl'

# Save the model using joblib
joblib.dump(final_model, model_filename)

print(f"Model saved as {model_filename}")

# END OF MODELLING

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import xgboost as xgb
# Assuming X_train, y_train, X_val, y_val, X_test, y_test are already defined
# Assuming xgb is imported as import xgboost as xgb

# Define the best parameters found during optimization
best_params = {
    'n_estimators': 1834,
    'max_depth': 11,
    'learning_rate': 0.004854856793781717,
    'subsample': 0.40566526847932777,
    'colsample_bytree': 0.3968091211824199,
    'gamma': 1.4684181309759596,
    'min_child_weight': 2,
    'reg_alpha': 0.9001406603742224,
    'reg_lambda': 0.23250578488239765,
    'random_state': 42
}

# Initialize the final model with the best hyperparameters
final_model = xgb.XGBRegressor(**best_params)

# Train the model on the training data
final_model.fit(X_train, y_train)

# Predict and evaluate on the training set
y_train_pred = final_model.predict(X_train)
train_metrics = evaluate_model(y_train, y_train_pred, "XGBoost-HPO", "Train")

# Predict and evaluate on the validation set
y_val_pred = final_model.predict(X_val)
val_metrics = evaluate_model(y_val, y_val_pred, "XGBoost-HPO", "Validation")

# Predict and evaluate on the test set
y_test_pred = final_model.predict(X_test)
test_metrics = evaluate_model(y_test, y_test_pred, "XGBoost-HPO", "Test")

# Combine metrics into a DataFrame for a comprehensive overview
metrics_df = pd.DataFrame([train_metrics, val_metrics, test_metrics])
print(metrics_df)

# Load Best XGBoost Model

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import joblib
import pandas as pd

# Load the saved model
model_filename = '/content/drive/MyDrive/windforecast/models/best_xgboost_model.pkl'
final_model = joblib.load(model_filename)

def evaluate_model(y_true, y_pred, model_name, dataset_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    max_ae = np.max(np.abs(y_true - y_pred))
    print(f"\nPerformance on {dataset_name} set:")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"R²: {r2:.4f}")
    print(f"MaxAE: {max_ae:.4f}")
    return {'Model': model_name, 'Dataset': dataset_name, 'RMSE': rmse, 'MAE': mae, 'R²': r2, 'MaxAE': max_ae}

# Assuming X_train, y_train, X_val, y_val, X_test, y_test are already defined

# Evaluate on the training set
y_train_pred = final_model.predict(X_train)
train_metrics = evaluate_model(y_train, y_train_pred, "XGBoost-HPO", "Train")

# Evaluate on the validation set
y_val_pred = final_model.predict(X_val)
val_metrics = evaluate_model(y_val, y_val_pred, "XGBoost-HPO", "Validation")

# Evaluate on the test set
y_test_pred = final_model.predict(X_test)
test_metrics = evaluate_model(y_test, y_test_pred, "XGBoost-HPO", "Test")

# Combine metrics into a DataFrame for a comprehensive overview
metrics_df = pd.DataFrame([train_metrics, val_metrics, test_metrics])
print(metrics_df)

In [ ]:
import pandas as pd
# Save both ground truths (y_train, y_val, y_test) and predictions (y_train_pred, y_val_pred, y_test_pred) as an excel file via a temp pandas dataframe
# Columns should be Set, Actual, Predictions. Set is categorical "Train", "Validation", "Test"
x= ['Train'] * len(y_train) + ['Validation'] * len(y_val) + ['Test'] * len(y_test)
temp_df = pd.DataFrame({
    'Set': x,
    'Actual': list(y_train) + list(y_val) + list(y_test),
    'Predictions': list(y_train_pred) + list(y_val_pred) + list(y_test_pred)
})
temp_df.to_excel('xgboost_predictions.xlsx', index=False)

In [ ]:
import pandas as pd
import os

# Assuming metrics_df is the DataFrame containing your newly computed metrics
# Path to the results file
results_path = '/content/drive/MyDrive/windforecast/model_results.csv'

# Load existing results if the file exists
try:
    existing_results_df = pd.read_csv(results_path)
    updated_results_df = pd.concat([existing_results_df, metrics_df], ignore_index=True)
except FileNotFoundError:
    updated_results_df = metrics_df
    print("Existing results file not found, saving current results only.")

# Save the updated results DataFrame to CSV
updated_results_df.to_csv(results_path, index=False)
print("Updated results saved to CSV.")

# GRAPHICAL VISUALIZATIONS

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
!pip install scienceplots
import scienceplots

# Data provided in CSV format (reconstructed for your environment)

# Load your data from a CSV file
data_dir = os.path.join(base_dir, 'data')
file_path = os.path.join(data_dir, 'results_windspeed.csv')  # Replace with the path to your CSV file
df = pd.read_csv(file_path)
df.head()

In [ ]:
# Make Dataset column composed of only three values: Train, Validation, Test
df['Dataset'] = df['Dataset'].apply(lambda x: 'Train' if x == 'Train' else ('Validation' if x == 'Validation' else 'Test'))
df.tail()

In [ ]:
# Plot comparison of RMSE scores across models
plt.figure(figsize=(14, 8))
sns.barplot(data=df, x='Model', y='RMSE', hue='Dataset', palette='muted')
plt.title('Model Performance Comparison (RMSE)')
plt.xticks(rotation=90, ha='right')
plt.ylabel('RMSE')
plt.xlabel('Model')
plt.legend(title='Dataset')
plt.tight_layout()
plt.savefig('performance_comparison.png')  # Save the plot as an image file
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Identify key models to focus on
selected_models = ['XGBoost-HPO', 'RandomForest', 'LightGBM', 'XGBoost']  # Customize this list
filtered_df = df[df['Model'].isin(selected_models)]

# Plot comparison of RMSE scores for selected models
plt.figure(figsize=(12, 6))
sns.barplot(data=filtered_df, x='Model', y='RMSE', hue='Dataset', palette='muted')
plt.title('Model Performance Comparison (RMSE)')
plt.xticks(rotation=45, ha='right')
plt.ylabel('RMSE')
plt.xlabel('Model')
plt.legend(title='Dataset')
plt.tight_layout()
plt.savefig('filtered_performance_comparison.png')  # Save the plot as an image file
plt.show()

In [ ]:
# Identify key models to focus on
selected_models = ['XGBoost-HPO', 'RandomForest', 'LightGBM', 'XGBoost']  # Customize this list
filtered_df = df[df['Model'].isin(selected_models)]

# Use scienceplots for better-looking plots
plt.style.use(['science', 'notebook', 'grid'])

# Plot comparison of RMSE scores for selected models
plt.figure(figsize=(12, 6), dpi=300)  # Set DPI for high resolution
sns.barplot(data=filtered_df, x='Model', y='RMSE', hue='Dataset', palette='muted')
plt.title('Model Performance Comparison (RMSE)')
plt.xticks(rotation=45, ha='right')
plt.ylabel('RMSE')
plt.xlabel('Model')
plt.legend(title='Dataset')
plt.tight_layout()

# Save the plot in both PNG and SVG formats
plt.savefig('filtered_performance_comparison.png', dpi=300, bbox_inches='tight')
plt.savefig('filtered_performance_comparison.svg', format='svg', bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns  # For additional plot types like histograms

# Set the professional parameters
plt.style.use([
    'science',
    'ieee',
    'grid',
    'notebook'
])

plt.rcParams.update({
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'lines.linewidth': 1.8,
    'axes.linewidth': 1.2,
    'grid.linewidth': 0.7,
    'legend.fontsize': 10,
    'legend.framealpha': 0.8,
    'legend.loc': 'best',
    'figure.dpi': 600,
    'figure.autolayout': True,
    'mathtext.fontset': 'stix',
    'font.family': 'serif',
    'font.serif': ['Times New Roman']
})

# Simulate dataset - replace this with actual data loading
np.random.seed(42)
dates = pd.date_range(start="2020-01-01", periods=100, freq="D")
wind_speed = np.random.normal(loc=10, scale=2, size=100).clip(0, 20)
wind_direction = np.random.uniform(0, 360, 100)
temperature = np.random.normal(loc=15, scale=5, size=100)

df = pd.DataFrame({
    'Date': dates,
    'Wind Speed': wind_speed,
    'Wind Direction': wind_direction,
    'Temperature': temperature
})

# Figure 1: Time series plot of wind speed and temperature
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(df['Date'], df['Wind Speed'], color='#004488', marker='o', markersize=5,
        markeredgecolor='w', markeredgewidth=0.8, label='Wind Speed')
ax.set_xlabel('Date')
ax.set_ylabel('Wind Speed (km/h)')
ax.set_title('Time Series of Wind Speed')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('wind_speed_time_series.png', dpi=600, bbox_inches='tight')
plt.savefig('wind_speed_time_series.eps', format='eps', bbox_inches='tight')
plt.show()

# Figure 2: Distribution of Wind Speed
fig, ax = plt.subplots(figsize=(6, 4))
sns.histplot(df['Wind Speed'], kde=True, color='#004488', ax=ax)
ax.set_xlabel('Wind Speed (km/h)')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Wind Speed')
plt.tight_layout()
plt.savefig('wind_speed_distribution.png', dpi=600, bbox_inches='tight')
plt.savefig('wind_speed_distribution.eps', format='eps', bbox_inches='tight')
plt.show()

# Figure 3: Correlation heatmap
corr = df[['Wind Speed', 'Wind Direction', 'Temperature']].corr()
fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix of Key Variables')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=600, bbox_inches='tight')
plt.savefig('correlation_heatmap.eps', format='eps', bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set base directory for Google Drive
base_dir = '/content/drive/MyDrive/windforecast/'
data_dir = os.path.join(base_dir, 'data')
results_dir = os.path.join(base_dir, 'results')
os.makedirs(results_dir, exist_ok=True)

# Set IEEE-compliant plotting style without LaTeX, avoiding specific font errors
plt.style.use(['science', 'ieee', 'no-latex'])
plt.rcParams.update({
    'font.size': 8,
    'axes.titlesize': 10,  # No title, kept for consistency
    'axes.labelsize': 8,
    'xtick.labelsize': 6,
    'ytick.labelsize': 6,
    'lines.linewidth': 1.2,
    'axes.linewidth': 0.8,
    'grid.linewidth': 0.5,
    'legend.fontsize': 6,
    'legend.title_fontsize': 7,  # Smaller legend title
    'legend.framealpha': 0.8,
    'legend.loc': 'center left',
    'figure.dpi': 600,
    'figure.figsize': (3.0, 2.2),
    'font.family': 'serif',
    'mathtext.fontset': 'dejavusans'
})

# Load data
file_path = os.path.join(data_dir, 'results_windspeed.csv')
df = pd.read_csv(file_path)

# Standardize Dataset column
df['Dataset'] = df['Dataset'].apply(lambda x: 'Train' if x == 'Train' else ('Validation' if x == 'Validation' else 'Test'))

# Filter to top-5 models plus Persistence, excluding XGBoost-Best
selected_models = ['XGBoost', 'LightGBM', 'RF', 'CatBoost', 'SVR', 'Persistence']
filtered_df = df[df['Model'].isin(['XGBoost', 'LightGBM', 'RandomForest', 'CatBoost', 'SVR', 'Persistence'])].copy()
# rename model name RandomForest to RF
filtered_df.loc[filtered_df['Model'] == 'RandomForest', 'Model'] = 'RF'

# Define Paul Tol’s muted color palette for six models
palette = {
    'Train': '#4477AA',      # Blue
    'Validation': '#EE6677', # Red
    'Test': '#228833'        # Green
}

# Create bar plot without title
fig, ax = plt.subplots()
sns.barplot(data=filtered_df, x='Model', y='RMSE', hue='Dataset', palette=palette, ax=ax)
ax.set_xlabel('Model')
ax.set_ylabel('RMSE (m/s)')
ax.tick_params(axis='x', rotation=45, labelleft=True)  # Move x-labels to left

# Move legend outside with smaller title
ax.legend(title='Dataset', bbox_to_anchor=(1.05, 1), loc='upper left')

# Save to Google Drive
plt.savefig(os.path.join(results_dir, 'performance_comparison_bar_plot_revised_v4.eps'), format='eps', bbox_inches='tight')
plt.savefig(os.path.join(results_dir, 'performance_comparison_bar_plot_revised_v4.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import matplotlib as mpl

# Set base directory for Google Drive
base_dir = '/content/drive/MyDrive/windforecast/'
data_dir = os.path.join(base_dir, 'data')
results_dir = os.path.join(base_dir, 'results')
os.makedirs(results_dir, exist_ok=True)

# Set IEEE-compliant plotting style without LaTeX, avoiding specific font errors
plt.style.use(['science', 'ieee', 'no-latex'])
plt.rcParams.update({
    'font.size': 8,
    'axes.titlesize': 10,  # No title, kept for consistency
    'axes.labelsize': 8,
    'xtick.labelsize': 6,
    'ytick.labelsize': 6,
    'lines.linewidth': 1.2,
    'axes.linewidth': 0.8,
    'grid.linewidth': 0.5,
    'legend.fontsize': 5,
    'legend.title_fontsize': 6,  # Smaller legend title
    'legend.framealpha': 0.8,
    'legend.loc': 'center left',
    'figure.dpi': 600,
    'figure.figsize': (3.0, 2.2),
    'font.family': 'serif',
    'mathtext.fontset': 'dejavusans'
})

# Load data
file_path = os.path.join(data_dir, 'results_windspeed.csv')
df = pd.read_csv(file_path)

# Standardize Dataset column
df['Dataset'] = df['Dataset'].apply(lambda x: 'Train' if x == 'Train' else ('Validation' if x == 'Validation' else 'Test'))

# Filter to top-5 models plus Persistence, excluding XGBoost-Best
selected_models = ['XGBoost', 'LightGBM', 'RF', 'CatBoost', 'SVR', 'Persistence']
filtered_df = df[df['Model'].isin(['XGBoost', 'LightGBM', 'RandomForest', 'CatBoost', 'SVR', 'Persistence'])].copy()
# Rename RandomForest to RF for brevity
filtered_df.loc[filtered_df['Model'] == 'RandomForest', 'Model'] = 'RF'

# Define Paul Tol’s muted color palette for six models
palette = {
    'Train': '#4477AA',      # Blue
    'Validation': '#EE6677', # Red
    'Test': '#228833'        # Green
}

# Create bar plot without title
fig, ax = plt.subplots()
sns.barplot(data=filtered_df, x='Model', y='RMSE', hue='Dataset', palette=palette, ax=ax)
ax.set_xlabel(None)
ax.set_ylabel('RMSE (m/s)')

# Move tick labels to the left using setp with negative x offset
plt.setp(ax.get_xticklabels(), rotation=45, ha='center')  # Align right for rotation, then shift left

# Apply precise left shift (5 points)
dx = -2  # Points to move left (negative = left)
dy = 0
import matplotlib.transforms as mtrans
offset = mtrans.ScaledTranslation(dx/72, dy/72, fig.dpi_scale_trans)

for label in ax.get_xticklabels():
    label.set_transform(label.get_transform() + offset)

# Move legend outside with smaller title
ax.legend(title='Dataset', bbox_to_anchor=(1.05, 1), loc='upper left')

# Save to Google Drive
plt.savefig(os.path.join(results_dir, 'performance_comparison_bar_plot_revised_v7.eps'), format='eps', bbox_inches='tight')
plt.savefig(os.path.join(results_dir, 'performance_comparison_bar_plot_revised_v7.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np

# Set base directory for Google Drive
base_dir = '/content/drive/MyDrive/windforecast/'
results_dir = os.path.join(base_dir, 'results')
os.makedirs(results_dir, exist_ok=True)

# Set IEEE-compliant plotting style without LaTeX
plt.style.use(['science', 'ieee', 'no-latex'])
plt.rcParams.update({
    'font.size': 8,
    'axes.labelsize': 8,
    'xtick.labelsize': 6,
    'ytick.labelsize': 6,
    'lines.linewidth': 1.2,
    'axes.linewidth': 0.8,
    'legend.fontsize': 6,
    'figure.dpi': 600,
    'figure.figsize': (3.3, 2.5),  # IEEE single-column width
    'font.family': 'serif',
    'mathtext.fontset': 'dejavusans'
})

# Generate dates for 2024 test set (adjust if your dates differ)
dates = pd.date_range(start='2024-01-01', periods=len(y_test), freq='D')

# Plot time series (replace with your y_test and y_test_pred)
fig, ax = plt.subplots()
ax.plot(dates, y_test, label='Actual', color='#4477AA', linewidth=1.0)
ax.plot(dates, y_test_pred, label='Predicted', color='#EE6677', linewidth=1.0, linestyle='--')
ax.set_xlabel(None)  # No title, use caption
ax.set_ylabel('Mean Wind Speed (m/s)')
ax.tick_params(axis='x', rotation=45, pad=0)  # Shift labels left slightly
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

# Save to Google Drive
plt.savefig(os.path.join(results_dir, 'time_series_plot.eps'), format='eps', bbox_inches='tight')
plt.savefig(os.path.join(results_dir, 'time_series_plot.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import xgboost as xgb
import numpy as np
import scienceplots

# Set base directory for Google Drive
base_dir = '/content/drive/MyDrive/windforecast/'
results_dir = os.path.join(base_dir, 'results')
os.makedirs(results_dir, exist_ok=True)

# Set IEEE-compliant plotting style without LaTeX
plt.style.use(['science', 'ieee', 'no-latex'])
plt.rcParams.update({
    'font.size': 8,
    'axes.labelsize': 8,
    'xtick.labelsize': 6,
    'ytick.labelsize': 6,
    'lines.linewidth': 1.2,
    'axes.linewidth': 0.8,
    'legend.fontsize': 6,
    'figure.dpi': 600,
    'figure.figsize': (3.3, 2.5),  # IEEE single-column width
    'font.family': 'serif',
    'mathtext.fontset': 'dejavusans'
})

# Load your trained model (adjust path as needed)
model_path = '/content/drive/MyDrive/windforecast/models/best_xgboost_model.pkl'
final_model = xgb.XGBRegressor()
final_model.load_model(model_path.replace('.pkl', '.json'))  # Load from JSON if saved

feature_names = X_train.columns.tolist()

# Get feature importances and create DataFrame
importances = final_model.feature_importances_
feature_importance = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feature_importance = feature_importance.nlargest(10, 'Importance')

# Format feature names: replace '_' with spaces and capitalize
feature_importance['Feature'] = feature_importance['Feature'].str.replace('_', ' ').str.title()

# Check if importances include negative values
has_negative = (importances < 0).any()
colors = ['darkorange' if imp >= 0 else '#EE6677' for imp in feature_importance['Importance']]  # Blue for positive, red for negative

# Create bar plot
fig, ax = plt.subplots()
ax.barh(feature_importance['Feature'], feature_importance['Importance'], color=colors)
ax.set_xlabel('Feature Importance')
ax.tick_params(axis='y', labelleft=True, rotation=45)  # Rotate y-ticks for space
ax.invert_yaxis()  # Highest importance at top

# Save to Google Drive
plt.savefig(os.path.join(results_dir, 'feature_importance_bar_chart.eps'), format='eps', bbox_inches='tight')
plt.savefig(os.path.join(results_dir, 'feature_importance_bar_chart.pdf'), bbox_inches='tight')
plt.show()

# Inform about importance values
print(f"Importances include negative values: {has_negative}")
if not has_negative:
    print("All importances are positive—color coding to blue only. No need to feel foolish! :)")

In [ ]:
feature_importances_xgboost = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feature_importances_xgboost.sort_values(by='Importance', ascending=False, inplace=True)
print(feature_importances_xgboost.head(10).values)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import xgboost as xgb

# Set base directory for Google Drive
base_dir = '/content/drive/MyDrive/windforecast/'
results_dir = os.path.join(base_dir, 'results')
os.makedirs(results_dir, exist_ok=True)

# Set IEEE-compliant plotting style without LaTeX
plt.style.use(['science', 'ieee', 'no-latex'])
plt.rcParams.update({
    'font.size': 8,
    'axes.labelsize': 8,
    'xtick.labelsize': 6,
    'ytick.labelsize': 6,
    'lines.linewidth': 1.2,
    'axes.linewidth': 0.8,
    'legend.fontsize': 6,
    'figure.dpi': 600,
    'figure.figsize': (3.3, 2.5),  # IEEE single-column width
    'font.family': 'serif',
    'mathtext.fontset': 'dejavusans'
})

# Assume results are from your previous fit (replace with actual results)
# Retrieve best parameters and train the final model
best_params = {'n_estimators': 1834, 'max_depth': 11, 'learning_rate': 0.004854856793781717, 'subsample': 0.40566526847932777, 'colsample_bytree': 0.3968091211824199, 'gamma': 1.4684181309759596, 'min_child_weight': 2, 'reg_alpha': 0.9001406603742224, 'reg_lambda': 0.23250578488239765}
print("Best hyperparameters:", best_params)
final_model = xgb.XGBRegressor(**best_params, random_state=42)
eval_set = [(X_train, y_train), (X_val, y_val)]
final_model.fit(X_train, y_train, eval_set=eval_set, verbose=False)
results = final_model.evals_result()

# Extract epochs and RMSE values
epochs = range(len(results['validation_0']['rmse']))
train_rmse = results['validation_0']['rmse']
val_rmse = results['validation_1']['rmse']

# Create learning curves plot
fig, ax = plt.subplots()
ax.plot(epochs, train_rmse, label='Train', color='#4477AA', linewidth=1.0)
ax.plot(epochs, val_rmse, label='Validation', color='#EE6677', linewidth=1.0, linestyle='--')
ax.set_xlabel('Number of Boosting Rounds')
ax.set_ylabel('RMSE (m/s)')
ax.tick_params(axis='x', rotation=0)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
# grid lines
ax.grid(True, linestyle='--', alpha=0.7)

# Save to Google Drive
plt.savefig(os.path.join(results_dir, 'learning_curves_plot.eps'), format='eps', bbox_inches='tight')
plt.savefig(os.path.join(results_dir, 'learning_curves_plot.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
X_train

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import numpy as np

# Set base directory for Google Drive
base_dir = '/content/drive/MyDrive/windforecast/'
results_dir = os.path.join(base_dir, 'results')
os.makedirs(results_dir, exist_ok=True)

# Set IEEE-compliant plotting style without LaTeX
plt.style.use(['science', 'ieee', 'no-latex'])
plt.rcParams.update({
    'font.size': 8,
    'axes.labelsize': 8,
    'xtick.labelsize': 6,
    'ytick.labelsize': 6,
    'lines.linewidth': 1.0,
    'axes.linewidth': 0.8,
    'legend.fontsize': 6,
    'figure.dpi': 600,
    'figure.figsize': (3.3, 2.5),  # IEEE single-column width
    'font.family': 'serif',
    'mathtext.fontset': 'dejavusans'
})

# Generate dates for 2024 test set (adjust if your dates differ)
dates = pd.date_range(start='2024-01-01', periods=len(y_test), freq='D')

# Compute residuals (replace with your actual y_test and y_test_pred)
residuals = y_test - y_test_pred  # Replace with your data

# Create subplot for ACF and PACF
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(3.3, 2.5), sharex=True)
plot_acf(residuals, ax=ax1, lags=20, title=None, color='#4477AA')
plot_pacf(residuals, ax=ax2, lags=20, title=None, color='#EE6677')
ax1.set_ylabel('ACF')
ax2.set_ylabel('PACF')
ax2.set_xlabel(None)  # No title, use caption
plt.tight_layout()

# Save to Google Drive
plt.savefig(os.path.join(results_dir, 'acfpacf_plot.eps'), format='eps', bbox_inches='tight')
plt.savefig(os.path.join(results_dir, 'acfpacf_plot.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import numpy as np

# Set base directory for Google Drive
base_dir = '/content/drive/MyDrive/windforecast/'
results_dir = os.path.join(base_dir, 'results')
os.makedirs(results_dir, exist_ok=True)

# Set IEEE-compliant plotting style without LaTeX
plt.style.use(['science', 'ieee', 'no-latex'])
plt.rcParams.update({
    'font.size': 8,
    'axes.labelsize': 8,
    'xtick.labelsize': 6,
    'ytick.labelsize': 6,
    'lines.linewidth': 1.0,
    'axes.linewidth': 0.8,
    'legend.fontsize': 6,
    'figure.dpi': 600,
    'figure.figsize': (3.3, 2.5),  # IEEE single-column width
    'font.family': 'serif',
    'mathtext.fontset': 'dejavusans'
})

# Generate dates for 2024 test set (adjust if your dates differ)
dates = pd.date_range(start='2024-01-01', periods=len(y_test), freq='D')

# Compute residuals (replace with your actual y_test and y_test_pred)
residuals = y_test - y_test_pred  # Replace with your data

# Create subplot for ACF and PACF with confidence intervals
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(3.3, 2.5), sharex=True, gridspec_kw={'hspace': 0.3})
plot_acf(residuals, ax=ax1, lags=20, alpha=0.05, title=None, color='#4477AA')  # 95% confidence intervals
plot_pacf(residuals, ax=ax2, lags=20, alpha=0.05, title=None, color='#EE6677')  # 95% confidence intervals
ax1.set_ylabel('ACF')
ax2.set_ylabel('PACF')
ax2.set_xlabel(None)  # No title, use caption
plt.tight_layout()

# Save to Google Drive
plt.savefig(os.path.join(results_dir, 'acfpacf_plot_refined.eps'), format='eps', bbox_inches='tight')
plt.savefig(os.path.join(results_dir, 'acfpacf_plot_refined.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import numpy as np

# Set base directory for Google Drive
base_dir = '/content/drive/MyDrive/windforecast/'
results_dir = os.path.join(base_dir, 'results')
os.makedirs(results_dir, exist_ok=True)

# Set IEEE-compliant plotting style without LaTeX
plt.style.use(['science', 'ieee', 'no-latex'])
plt.rcParams.update({
    'font.size': 8,
    'axes.labelsize': 8,
    'xtick.labelsize': 6,
    'ytick.labelsize': 6,
    'lines.linewidth': 1.0,
    'axes.linewidth': 0.8,
    'legend.fontsize': 6,
    'figure.dpi': 600,
    'figure.figsize': (6.6, 5.0),  # Doubled size for visibility
    'font.family': 'serif',
    'mathtext.fontset': 'dejavusans'
})

# Generate dates for 2024 test set (adjust if your dates differ)
dates = pd.date_range(start='2024-01-01', periods=len(y_test), freq='D')

# Compute residuals (replace with your actual y_test and y_test_pred)
residuals = y_test - y_test_pred  # Replace with your data

# Create subplot for ACF and PACF with enhanced visibility
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6.6, 5.0), sharex=True, gridspec_kw={'hspace': 0.5})
plot_acf(residuals, ax=ax1, lags=20, alpha=0.05, title=None, color='#4477AA', marker='o')  # Added marker
plot_pacf(residuals, ax=ax2, lags=20, alpha=0.05, title=None, color='#EE6677', marker='o')  # Added marker
ax1.set_ylabel('ACF')
ax2.set_ylabel('PACF')
ax2.set_xlabel(None)  # No title, use caption
plt.tight_layout()

# Save to Google Drive
plt.savefig(os.path.join(results_dir, 'acfpacf_plot_enhanced.eps'), format='eps', bbox_inches='tight')
plt.savefig(os.path.join(results_dir, 'acfpacf_plot_enhanced.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
X_train.shape, X_train_cv.shape

In [ ]:
import shap
import matplotlib.pyplot as plt
import os
import xgboost as xgb
import pickle
import pandas as pd
import numpy as np  # Added for background sampling

# Set base directory for Google Drive
base_dir = '/content/drive/MyDrive/windforecast/'
results_dir = os.path.join(base_dir, 'results')
os.makedirs(results_dir, exist_ok=True)

# Set IEEE-compliant plotting style without LaTeX
plt.style.use(['science', 'ieee', 'no-latex'])
plt.rcParams.update({
    'font.size': 8,
    'axes.labelsize': 8,
    'xtick.labelsize': 6,
    'ytick.labelsize': 6,
    'lines.linewidth': 1.0,
    'axes.linewidth': 0.8,
    'legend.fontsize': 6,
    'figure.dpi': 600,
    'figure.figsize': (3.3, 2.5),  # IEEE single-column width
    'font.family': 'serif',
    'mathtext.fontset': 'dejavusans'
})

# 1. FIXED MODEL LOADING
model_path = '/content/drive/MyDrive/windforecast/models/best_xgboost_model.pkl'
try:
    # Load model directly from JSON (XGBoost native format)
    final_model = xgb.XGBRegressor()
    final_model.load_model(model_path.replace('.pkl', '.json'))
    print("Model loaded successfully.")
except Exception as e:
    print(f"Error loading model: {e}")
    # Add fallback or exit if needed

background_data = X_train

# 4. FIXED SHAP COMPUTATION
try:
    # Create explainer with model object (not .predict)
    explainer = shap.TreeExplainer(final_model, background_data)

    # 5. USE CORRECT SHAP VALUE EXTRACTION
    shap_values = explainer(X_test)

    # 6. HANDLE FEATURE NAMES PROPERLY
    feature_names = [f.replace('_', ' ').title() for f in X_test.columns]

    print("SHAP values computed successfully.")
except Exception as e:
    print(f"Error in SHAP computation: {e}")
    raise

# 7. FIXED PLOTTING CODE
try:
    plt.figure()

    # Create bee-swarm plot with corrected parameters
    shap.summary_plot(
        shap_values.values,  # Extract numpy array from Explanation object
        X_test,              # Use original feature names for calculation
        feature_names=feature_names,  # Pass formatted names separately
        plot_type="dot",     # 'dot' is the actual name for bee-swarm
        show=False,
        color_bar=True,
        max_display=10,
    )

    plt.xlabel("SHAP Value (Impact on Model Output)")
    plt.tight_layout()

    # Save to Google Drive
    plt.savefig(os.path.join(results_dir, 'shap_beeswarm_plot.eps'), format='eps', bbox_inches='tight')
    plt.savefig(os.path.join(results_dir, 'shap_beeswarm_plot.pdf'), bbox_inches='tight')
    plt.show()
    print("Plot saved successfully.")
except Exception as e:
    print(f"Error creating plot: {e}")